In [22]:
import torch
import math
import torch.nn.functional as F
import numpy as np
import random
import os

def set_seed(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed) # for multi-GPU
    
    # Force deterministic algorithms
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False




In [55]:
B = 6
T = 6
n_heads = 12
n_embd = 768
set_seed()

temp = torch.tril(torch.ones(24, 24)).view(1, 1, 24, 24)
qkv = torch.randint(-256, 255, (B, T, 3*n_embd)).to(float)
print("qkv shape: ", qkv.shape)

q, k, v = qkv.split(n_embd, dim=2)
print("1. q shape: ", q.shape)

q = q.view(B, T, n_heads, n_embd // n_heads).transpose(1, 2)
k = k.view(B, T, n_heads, n_embd // n_heads).transpose(1, 2)
v = v.view(B, T, n_heads, n_embd // n_heads).transpose(1, 2)
print("2. q shape: ", q.shape)

attn = (q @ k.transpose(-1, -2)) * (1 / math.sqrt(k.shape[-1]))
print("3. attention shape: ", attn.shape) # B, T, n_embd, n_embd

attn = attn.masked_fill(temp[:, :, :T, :T] == 0, float('-inf'))
# attn = F.softmax(attn, dim=-1)

# y1 = attn @ v
# y2 = F.scaled_dot_product_attention(q, k, v, is_causal=True)

# torch.allclose(y1, y2)




qkv shape:  torch.Size([6, 6, 2304])
1. q shape:  torch.Size([6, 6, 768])
2. q shape:  torch.Size([6, 12, 6, 64])
3. attention shape:  torch.Size([6, 12, 6, 6])


In [56]:
attn[0, 0, :, :]

tensor([[ 17018.6250,        -inf,        -inf,        -inf,        -inf,
                -inf],
        [ -2985.2500,  10561.1250,        -inf,        -inf,        -inf,
                -inf],
        [  6524.6250,   3103.2500, -24048.7500,        -inf,        -inf,
                -inf],
        [ 19219.5000, -32286.5000, -18230.8750, -12468.3750,        -inf,
                -inf],
        [  1171.5000, -28463.3750,  14880.7500,  -8841.6250, -16592.3750,
                -inf],
        [-25120.1250, -17654.0000,  25264.1250,  24192.5000,   7871.2500,
           3059.3750]], dtype=torch.float64)